## **import lib**

In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage.morphology import skeletonize, binary_dilation, disk
import segmentation_models_pytorch as smp

c:\Users\MOHAMMED_PC\miniconda3\envs\myenv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## **Load Model**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
model = smp.Unet(
    encoder_name="efficientnet-b3",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None,
).to(device)
 
WEIGHTS_PATH = "../models/best_unet_vessels.pth"
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
    model.eval()
else:
    raise FileNotFoundError(f"NO model found: {WEIGHTS_PATH}")

## **Load image**

In [ ]:
IMAGE_PATH = r"C:\\Users\\MOHAMMED_PC\\Downloads\\IDRiD_001.jpg"  
 
mean = np.array([0.6129, 0.2364, 0.1281], dtype=np.float32)
std  = np.array([0.3016, 0.1446, 0.0837], dtype=np.float32)
 
orig_bgr = cv2.imread(IMAGE_PATH)
if orig_bgr is None:
    raise FileNotFoundError(f"Can Not read the image: {IMAGE_PATH}")
 
orig_rgb      = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
orig_512      = cv2.resize(orig_bgr, (512, 512), interpolation=cv2.INTER_CUBIC)
orig_512_rgb  = cv2.cvtColor(orig_512, cv2.COLOR_BGR2RGB)
 
# CLAHE  just on green channel
green_ch      = orig_bgr[:, :, 1]
clahe         = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced_green = clahe.apply(green_ch)
resized_green  = cv2.resize(enhanced_green, (512, 512), interpolation=cv2.INTER_CUBIC)
 
# transform to Tensor
img_t = resized_green.astype(np.float32) / 255.0
img_t = np.stack([img_t] * 3, axis=-1)
img_t = (img_t - mean) / std
img_t = torch.from_numpy(img_t).permute(2, 0, 1).unsqueeze(0).to(device)
 

## **predict Model**

In [ ]:
with torch.no_grad():
    output     = model(img_t)
    prob_map   = torch.sigmoid(output).cpu().numpy()[0, 0]   
    pred_mask  = (prob_map > 0.5).astype(np.uint8)           
 

## **Skeleton Algorithm**

In [ ]:
def detect_endpoints(skeleton: np.ndarray) -> np.ndarray:

    kernel = np.ones((3, 3), dtype=np.uint8)
    kernel[1, 1] = 0  
    neighbor_count = cv2.filter2D(skeleton.astype(np.uint8), -1, kernel)

    endpoints = (skeleton == 1) & (neighbor_count == 1)
    return endpoints.astype(np.uint8)
 
 
def bridge_gaps(skeleton: np.ndarray,
                endpoints: np.ndarray,
                max_gap_px: int = 7,
                max_angle_diff_deg: float = 30.0) -> np.ndarray:
    
    bridged = skeleton.copy()
    coords  = np.column_stack(np.where(endpoints == 1))   
 
    if len(coords) < 2:
        return bridged
 
    def local_direction(skel, r, c, radius=5):
        r0, r1 = max(0, r - radius), min(skel.shape[0], r + radius + 1)
        c0, c1 = max(0, c - radius), min(skel.shape[1], c + radius + 1)
        patch   = skel[r0:r1, c0:c1]
        pts     = np.column_stack(np.where(patch == 1))
        if len(pts) < 2:
            return None
        pts_centered = pts - pts.mean(axis=0)
        _, _, vt = np.linalg.svd(pts_centered)
        return vt[0]   
 
    used = set()
    for i, (r1, c1) in enumerate(coords):
        if i in used:
            continue
        dir1 = local_direction(skeleton, r1, c1)
 
        for j, (r2, c2) in enumerate(coords):
            if j <= i or j in used:
                continue
 
            dist = np.hypot(r2 - r1, c2 - c1)
            if dist > max_gap_px:
                continue
 
            if dir1 is not None:
                dir2 = local_direction(skeleton, r2, c2)
                if dir2 is not None:
                    cos_angle = np.clip(abs(np.dot(dir1, dir2)), 0, 1)
                    angle_diff = np.degrees(np.arccos(cos_angle))
                    if angle_diff > max_angle_diff_deg:
                        continue
 
            line_pts = np.column_stack([
                np.linspace(r1, r2, num=int(dist) + 2, dtype=int),
                np.linspace(c1, c2, num=int(dist) + 2, dtype=int),
            ])
            for rp, cp in line_pts:
                if 0 <= rp < bridged.shape[0] and 0 <= cp < bridged.shape[1]:
                    bridged[rp, cp] = 1
 
            used.add(i)
            used.add(j)
            break   
 
    return bridged
 
 
def apply_skeletonization_pipeline(pred_mask: np.ndarray,
                                   dilation_radius: int = 1,
                                   max_gap_px: int = 7,
                                   max_angle_diff_deg: float = 30.0) -> np.ndarray:
    """
    خط معالجة الهيكلة الكامل:
      1. استخراج الهيكل (Zhang-Suen)
      2. اكتشاف نقاط النهاية
      3. ربط الفجوات الصغيرة
      4. توسيع بسيط لاستعادة سمك معقول
      5. دمج الجسور مع القناع الأصلي
 
    يُعيد قناعاً ثنائياً محسّناً (uint8 ، قيم 0 أو 1).
    """
    mask_bool = pred_mask.astype(bool)
 
    # ── المرحلة 1: الهيكلة ──────────────────────────────────
    skeleton = skeletonize(mask_bool).astype(np.uint8)
 
    # ── المرحلة 2: اكتشاف نقاط النهاية ─────────────────────
    endpoints = detect_endpoints(skeleton)
    n_endpoints = int(endpoints.sum())
    print(f"   🔍 نقاط نهاية مُكتشَفة: {n_endpoints}")
 
    # ── المرحلة 3: ربط الفجوات ──────────────────────────────
    bridged_skeleton = bridge_gaps(skeleton, endpoints,
                                   max_gap_px=max_gap_px,
                                   max_angle_diff_deg=max_angle_diff_deg)
    dilated_bridge = binary_dilation(
        bridged_skeleton.astype(bool),
        footprint=disk(dilation_radius)
    ).astype(np.uint8)
    enhanced_mask = np.clip(pred_mask.astype(np.uint8) + dilated_bridge, 0, 1)
 
    return enhanced_mask, skeleton, bridged_skeleton
print("⚙️  جاري تطبيق خوارزمية الهيكلة …")
enhanced_mask, skeleton, bridged_skeleton = apply_skeletonization_pipeline(
    pred_mask,
    dilation_radius=1,      # توسّع خفيف جداً — غيّر لـ 2 إن احتجت سُمكاً أكبر
    max_gap_px=20,           # أقصى فجوة تُربط (بكسل)
    max_angle_diff_deg=90,  # أقصى فرق زاوي
)
print("✅  اكتملت الهيكلة!")


⚙️  جاري تطبيق خوارزمية الهيكلة …
   🔍 نقاط نهاية مُكتشَفة: 141
✅  اكتملت الهيكلة!


## **Overlay**

In [ ]:
def make_overlay(base_rgb: np.ndarray, mask: np.ndarray,
                 color=(0, 255, 0), alpha=0.35) -> np.ndarray:
    colored = np.zeros_like(base_rgb)
    colored[mask == 1] = color
    return cv2.addWeighted(base_rgb, 1 - alpha, colored, alpha, 0)
 
overlay_basic    = make_overlay(orig_512_rgb, pred_mask)
overlay_enhanced = make_overlay(orig_512_rgb, enhanced_mask)


## **show final result**

In [ ]:
fig = plt.figure(figsize=(20, 10))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.08)
 
panels = [
    (gs[0, 0], orig_512_rgb,       "Orginal Image ",          "gray",  False),
    (gs[0, 1], pred_mask,          "Orginal Mask","gray",  True),
    (gs[0, 2], overlay_basic,      "Overlay Orginal",         None,    False),
    (gs[0, 3], enhanced_mask,      "skeleton mask","gray",  True),
    (gs[1, 0], skeleton,           "(Skeleton)",        "hot",   True),
    (gs[1, 1], bridged_skeleton,   "Skeleton after applay",  "hot",   True),
    (gs[1, 2], overlay_enhanced,   "Overlay Skeleton",        None,    False),
    (gs[1, 3],
     (enhanced_mask.astype(int) - pred_mask.astype(int)).clip(0),
     "  (new bidge)", "cool", True),
]
 
for spec, img, title, cmap, is_binary in panels:
    ax = fig.add_subplot(spec)
    ax.set_facecolor('#0d1117')
 
    if img.ndim == 2:
        ax.imshow(img, cmap=cmap, vmin=0, vmax=1 if is_binary else img.max())
    else:
        ax.imshow(img)
 
    ax.set_title(title, color='#e6edf3', fontsize=10, pad=6,
                 fontfamily='DejaVu Sans')
    ax.axis('off')
 
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('#30363d')
        spine.set_linewidth(0.8)
 
fig.suptitle(
    "comparing",
    color='#58a6ff', fontsize=14, fontweight='bold', y=0.98
)
 
plt.savefig("vessel_skeleton_analysis.png", dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print("💾 the image was saved: vessel_skeleton_analysis.png")
 


NameError: name 'plt' is not defined